In [ ]:
import {torus, tensor} from '/torus/torus.js';
import {nn, Linear} from '/torus/neuro-module/neuro-module.js';

torus.USE_GPU = true
torus.USE_SCALAR_CACHE = true

In [ ]:
// _element_wise
    torus._element_wise = function ($ = {forward: '', forwardGPU: '', backward: '', backwardGPU: '', label: ''}, other){
        let key = $.forward + ': ' + (other?.shape? `(${other.shape})`: (other ?? '-')); 
        let out = torus.get_out(this, key);
        if (!out){
            let shape = this.shape;
            let src = [this];
            if (other?.shape) {  //other -- tensor
                src.push(other);
                shape = torus.get_broadcast_shapes(this, other);
            }
            out = tensor.from(new torus.DEFAULT_TYPE(shape.mul() || 1))._shape(shape)._src(src);
            torus.set_out(this, out, key);
            let out_info = out.shape_info.filter(dim_info => dim_info.size !== 1);   //Оси с длинной 1 в формировании реального индекса не участвуют
            let this_info = this.shape_info.filter(dim_info => dim_info.size !== 1);
            let other_info = other?.shape_info?.filter(dim_info => dim_info.size !== 1);
            if(other && other?.length>1 && this.length>1 && !torus._shapes_are_equal(this, other)) { // Если необходимо для каждого операнда рассчитывать собственный индекс выборки, пытаемся объединить подходящие оси
                const squeeze = (tensor_info) => {
                    tensor_info = structuredClone(tensor_info);
                    let squeeze_axes1 = tensor_info.filter(d_info => {   // Старшие оси - претенденты на объединение
                        return d_info.char.charCodeAt(0) > this_info[0].char.charCodeAt(0) || d_info.char.charCodeAt(0) > other_info[0].char.charCodeAt(0);
                    });
                    let squeeze_axes2 = tensor_info.filter(d_info => {   // Внутренние оси - претенденты на объединение
                        return this_info.some(di => d_info.char === di.char) && other_info.some(di => d_info.char === di.char);
                    });
                    [squeeze_axes1, squeeze_axes2].forEach(squeeze_axes => {
                        for (let i = 0; i <= squeeze_axes.length-2; i++) { // Объединяем соседние оси, чтобы не рассчитывать лишние индексы
                            if (squeeze_axes[i+1].idx - squeeze_axes[i].idx === 1 ) {
                                let info_0 = tensor_info.find(v => v.idx === squeeze_axes[i].idx);
                                let info_1 = tensor_info.find(v => v.idx === squeeze_axes[i+1].idx);
                                info_1.size *= info_0.size;
                                info_1.char += info_0.char;
                                tensor_info = tensor_info.filter(v => v.idx !== squeeze_axes[i].idx);
                            }
                        }
                    });
                    return tensor_info;
                }
                out_info = squeeze(out_info);
                let this_info_tmp = squeeze(this_info);
                other_info = squeeze(other_info);
                this_info = this_info_tmp;
            }
            if (Number.isFinite(other)) {   // Если второй операнд является числом, вставляем его напрямую в расчётные формулы
                $ = structuredClone($);
                if (Number.isInteger(other))
                    other = other + '.';
                $.forwardGPU = $.forwardGPU.replaceAll('x1', other);
                if ($.backwardGPU_0)
                    $.backwardGPU_0 = $.backwardGPU_0.replaceAll('x1', other);
                if ($.backwardGPU)
                    $.backwardGPU = $.backwardGPU.replaceAll('x1', other);
                $.forward = $.forward.replace(/, *x1/, '').replaceAll('x1', other);
                if ($.backward_0)
                    $.backward_0 = $.backward_0.replace(/, *x1/, '').replaceAll('x1', other);
                if ($.backward)
                    $.backward = $.backward.replace(/, *x1/, '').replaceAll('x1', other);
                other = undefined;
            }
            if (torus.USE_GPU){
                let wg = out.gpu_compute_info;
                let cb;
                if (other === undefined) {   // Если имеется только один операнд, то всё очень просто
                    cb = new torus.ShaderBuilder(
                        `// element_wise FORWARD`,
                        `@group(0) @binding(0) var<storage, read> self_data: array<${this.gpuType}>;`,
                        `@group(0) @binding(1) var<storage, read_write> out_data: array<${out.gpuType}>;`,
                        `fn calculate(x0: ${this.gpuType}) -> ${out.gpuType} {`,
                            $.forwardGPU?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                        `}`,
                        `@compute @workgroup_size(${wg.size})`,
                        `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                        wg.code,
                            `    out_data[idx] = calculate(self_data[idx]);`,
                        `}`
                    );
                }
                else if (!other?.shape || other.length===1 || this.length===1 || torus._shapes_are_equal(this, other)) {   // Если достаточно простой линейной выборки данных 
                    cb = new torus.ShaderBuilder(
                        `// element_wise FORWARD`,
                        `@group(0) @binding(0) var<storage, read> self_data: array<${this.gpuType}>;`,
                        (other?.shape)? `@group(0) @binding(1) var<storage, read> other_data: array<${other.gpuType}>;`: '',
                        `@group(0) @binding(${src.length}) var<storage, read_write> out_data: array<${out.gpuType}>;`,
                        `fn calculate(x0: ${this.gpuType}, x1: ${other?.gpuType || out.gpuType}) -> ${out.gpuType} {`,
                            $.forwardGPU?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                        `}`,
                        `@compute @workgroup_size(${wg.size})`,
                        `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                        wg.code,
                        `    out_data[idx] = calculate(self_data[${this.length===1? 0: 'idx'}], ${!other?.shape? other: (other.length===1? 'other_data[0]': 'other_data[idx]')});`,
                        `}`
                    );
                }
                else {  // Если необходимо для каждого операнда рассчитывать собственный индекс выборки
                    cb = new torus.ShaderBuilder(
                        `// element_wise FORWARD`,
                        `@group(0) @binding(0) var<storage, read> self_data: array<${this.gpuType}>;`,
                        `@group(0) @binding(1) var<storage, read> other_data: array<${other.gpuType}>;`,
                        `@group(0) @binding(2) var<storage, read_write> out_data: array<${out.gpuType}>;`,
                        `fn calculate(x0: ${this.gpuType}, x1: ${other.gpuType}) -> ${out.gpuType} {`,
                            $.forwardGPU?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                        `}`,
                        `@compute @workgroup_size(${wg.size})`,
                        `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                        wg.code,
                        out_info.map(di => {
                            if (this_info.some(v=>v.char===di.char) && this.length!==out.length || other_info.some(v=>v.char===di.char) && other.length!==out.length)
                                return `    let idx_${di.char} = ${di.stride===1? `idx % ${di.size}`: `(idx / ${di.stride}) % ${di.size}`};`;
                            return [];
                        }),
                        [this_info, other_info].map((shape_info, i) => {
                            if (shape_info.length === out_info.length)
                                return `    let idx_${i} = idx;`;
                            return `    let idx_${i} = ${shape_info.map(di => di.stride === 1? `idx_${di.char}`:`idx_${di.char} * ${di.stride}`).join(' + ')};`
                        }),
                        `    out_data[idx] = calculate(self_data[idx_0], other_data[idx_1]);`,
                        `}`
                    );
                }
                let shader = cb.shader;
// >cb.code                
                out._fwd = (other)=>{
                    let src = [this];
                    if(other?.shape)
                        src.push(other);                    
                    torus.compute(shader, [...src, out], wg.count);
                    return out._src(src);
                }
                
                if (src.some(t=>t.allowGrad)) {
                    const the_same = this.data === other?.data;
                    let cb;
                    if (other === undefined) {   // Если имеется только один операнд, то всё очень просто
                        cb = new torus.ShaderBuilder(
                            `// element_wise BACK`,
                            `@group(0) @binding(0) var<storage, read> out_grad: array<${out.grad.gpuType}>;`,
                            `@group(0) @binding(1) var<storage, read> self_data: array<${this.gpuType}>;`,
                            `@group(0) @binding(2) var<storage, read_write> self_grad: array<${this.grad.gpuType}>;`,
                            `fn fn_self(x0: ${this.gpuType}) -> ${out.gpuType} {`,
                                ($.backwardGPU_0 || $.backwardGPU)?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                            `}`,
                            `@compute @workgroup_size(${wg.size})`,
                            `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                            wg.code,
                            `    self_grad[idx] += out_grad[idx] * fn_self(self_data[idx]);`,
                            `}`
                        );
                    }
                    else if (!other?.shape || other.length===1 || this.length===1 || torus._shapes_are_equal(this, other)) {   // Если достаточно простой линейной выборки данных 
                        cb = new torus.ShaderBuilder(
                            `// element_wise BACK`,
                            `@group(0) @binding(0) var<storage, read> out_grad: array<${out.grad.gpuType}>;`,
                            `@group(0) @binding(1) var<storage, read> self_data: array<${this.gpuType}>;`,
                            (()=>{    //Объявляем буфера для данных второго параметра (если он тензор) и градиентов (если они нужны)
                                const code = [];
                                let binding_idx = 2;
                                if (this.allowGrad) {
                                    code.push(`@group(0) @binding(${binding_idx}) var<storage, read_write> self_grad: array<${this.grad.gpuType}>;`);
                                    binding_idx++;
                                }
                                if (!the_same && other?.shape) {
                                    code.push(`@group(0) @binding(${binding_idx}) var<storage, read> other_data: array<${other.gpuType}>;`);
                                    binding_idx++;
                                    if (other?.allowGrad)
                                        code.push(`@group(0) @binding(${binding_idx}) var<storage, read_write> other_grad: array<${other.grad.gpuType}>;`);
                                }
                                return code;
                            })(),
                            (this.allowGrad)? [
                                `fn fn_self(x0: ${this.gpuType}, x1: ${other?.gpuType || out.gpuType}) -> ${out.gpuType} {`,
                                    ($.backwardGPU_0 || $.backwardGPU)?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                                `}`
                            ]: '',
                            (other?.allowGrad)? [
                                `fn fn_other(x0: ${this.gpuType}, x1: ${other?.gpuType || out.gpuType}) -> ${out.gpuType} {`,
                                    ($.backwardGPU_1 || $.backwardGPU)?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                                `}`
                            ]: '',
                            `@compute @workgroup_size(${wg.size})`,
                            `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                            wg.code,
                            (this.allowGrad || other.length===out.length && other.allowGrad)?
                                                    `    let x0 = self_data[${this.length===1? 0: 'idx'}];`: [],
                            (()=>{
                                 if (the_same) return [];
                                 if (!(other.allowGrad || this.length===out.length && this.allowGrad)) return [];
                                 if (!other?.shape) return `    let x1 = ${out.gpuType}(${other});`;
                                 return `    let x1 = other_data[${other.length===1? 0: 'idx'}];`
                            })(),
                            (()=>{   // Рассчитываем и сохраняем градиенты
                                if (the_same)
                                    return `    self_grad[idx] += out_grad[idx] * (fn_self(x0, x0) + fn_other(x0, x0));`;
                                return [this, other].map((t, i) => {
                                    if (!t?.allowGrad) return [];
                                    let t_name = (i===0)? 'self': 'other';
                                    if (t.length === out.length)
                                        return `    ${t_name}_grad[idx] += out_grad[idx] * fn_${t_name}(x0, x1);`;
                                    return  [
                                            `    if (idx == 0) {`,
                                            `        for (var i = 0u; i < ${out.length}u; i++) {`,
                                            `            ${t_name}_grad[0] += out_grad[i] * fn_${t_name}(${i===0? 'x0, other_data[i]': 'self_data[i], x1'});`,
                                            `        }`,
                                            `    }`,
                                            ];
                                });
                            })(),
                            `}`
                        );
                    }
                    else {  // Если необходимо для каждого операнда рассчитывать собственный индекс выборки
                        const this_broadcast_axes = out_info.filter(dim_info => !this_info.some(di => dim_info.char === di.char));
                        const other_broadcast_axes = out_info.filter(dim_info => !other_info.some(di => dim_info.char === di.char));
                        const tab = '    ';
                        cb = new torus.ShaderBuilder(
                            `// element_wise BACK`,
                            `@group(0) @binding(0) var<storage, read> out_grad: array<${out.grad.gpuType}>;`,
                            `@group(0) @binding(1) var<storage, read> self_data: array<${this.gpuType}>;`,
                            (()=>{    //Объявляем буфера для данных второго параметра и градиентов (если они нужны)
                                const code = [];
                                let binding_idx = 2;
                                if (this.allowGrad) {
                                    code.push(`@group(0) @binding(${binding_idx}) var<storage, read_write> self_grad: array<${this.grad.gpuType}>;`);
                                    binding_idx++;
                                }
                                code.push(`@group(0) @binding(${binding_idx}) var<storage, read> other_data: array<${other.gpuType}>;`);
                                binding_idx++;
                                if (other.allowGrad) {
                                    code.push(`@group(0) @binding(${binding_idx}) var<storage, read_write> other_grad: array<${other.grad.gpuType}>;`);
                                }
                                return code;
                            })(),
                            (this.allowGrad)? [
                                `fn fn_self(x0: ${this.gpuType}, x1: ${other.gpuType}) -> ${out.gpuType} {`,
                                    ($.backwardGPU_0 || $.backwardGPU)?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                                `}`
                            ]: [],
                            (other.allowGrad)? [
                                `fn fn_other(x0: ${this.gpuType}, x1: ${other.gpuType}) -> ${out.gpuType} {`,
                                    ($.backwardGPU_1 || $.backwardGPU)?.split('\n').map(s => '    ' + s).join('\n'),  // Достаточно $.forwardGPU, остальное задаёт отступ тела метода
                                `}`
                            ]: [],
                            `@compute @workgroup_size(${wg.size})`,
                            `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                            wg.code,
                            out_info.map(di => {
                                if (this_info.some(v=>v.char===di.char) && (this.length!==out.length || other.allowGrad) || 
                                            other_info.some(v=>v.char===di.char) && (other.length!==out.length || this.allowGrad))
                                    return `    let idx_${di.char} = ${di.stride===1? `idx % ${di.size}`: `(idx / ${di.stride}) % ${di.size}`};`;
                                return [];
                            }),
                            [this, other].map((t, i, tensors) => {
                                let s_info = (i===0)? this_info: other_info;
                                if (t.allowGrad || tensors[i^1].length===out.length && tensors[i^1].allowGrad)
                                    return [
                                        t.length===out.length? `    let idx_${i} = idx;`:
                                            `    let idx_${i} = ${s_info.map(di => di.stride === 1? `idx_${di.char}`:`idx_${di.char} * ${di.stride}`).join(' + ')};`,
                                        `    let x${i} = ${i===0? 'self': 'other'}_data[idx_${i}];`
                                    ];
                                return [];
                            }),
                            (this.length===out.length && this.allowGrad || other.length===out.length && other.allowGrad)? `    let grad = out_grad[idx];`: [],
                            [this, other].map((t, i, tensors) => {
                                if (!t.allowGrad) return [];
                                let t_name = (i===0)? 'self': 'other';
                                if (t.length === out.length)
                                    return `    ${t_name}_grad[idx_${i}] += grad * fn_${t_name}(x0, x1);`;
                                let broadcast_axes = (i===0)? this_broadcast_axes: other_broadcast_axes;
                                const out_not_broadcast_axes = out_info.filter(dim_info => !broadcast_axes.some(di => dim_info.char === di.char));
                                let s_info = [this_info, other_info];
                                let tabs = tab.repeat(broadcast_axes.length + 2);
                                let out_start_idx_code = out_not_broadcast_axes.map(di => `idx_${di.char} ${di.stride > 1? `* ${di.stride}`: ''}`).join(' + ');
                                let input_crossing_axes = s_info[i^1].filter(d0 => s_info[i].some(d1 => d0.char === d1.char));
                                let in_start_idx_code = input_crossing_axes.map(di => `idx_${di.char} ${di.stride > 1? `* ${di.stride}`: ''}`).join(' + ');
                                return [
                                    `    if (${broadcast_axes.map(di => `idx_${di.char} == 0`).join(' && ')}) {`,
                                    broadcast_axes.map((di, j, d_info) => [
                                        tab.repeat(j+2) + `var idx_o_${di.char} = ${j===0? out_start_idx_code: `idx_o_${d_info[j-1].char}`};`,
                                        tab.repeat(j+2) + (() => {
                                            if (tensors[i^1].length === out.length) return '';
                                            return `var idx_i_${di.char} = ${(j===0? (input_crossing_axes.length? in_start_idx_code: 0): `idx_i_${d_info[j-1].char}`)};`;
                                        })(),
                                        tab.repeat(j+2) + `for (var idx_${di.char} = 0u; idx_${di.char} < ${di.size}; idx_${di.char}++) {`,
                                    ]),
                                    (() => {
                                        const idx_i_name = (tensors[i^1].length === out.length)? `idx_o_${broadcast_axes.last.char}`: `idx_i_${broadcast_axes.last.char}`;
                                        return tabs + `${t_name}_grad[idx_${i}] += out_grad[idx_o_${broadcast_axes.last.char}] * fn_${t_name}(${i===0? `x0, other_data[${idx_i_name}]`: `self_data[${idx_i_name}], x1`});`;
                                    })(),
                                    broadcast_axes.map((di, j, d_info) => [
                                        tab.repeat(j+3) + `idx_o_${di.char} += ${out_info.find(inf => inf.char === di.char).stride};`,
                                        tab.repeat(j+3) + (tensors[i^1].length===out.length? '' : `idx_i_${di.char} += ${s_info[i^1].find(inf => inf.char === di.char).stride};`),
                                        tab.repeat(j+2) + `}`
                                    ]).reverse(),
                                    `    }`
                                ];
                            }),
                            `}`
                        );
                    }
                    let shader = cb.shader;
// >cb.code
                    out._back = ()=>{
                        const tensors = [];
                        if (this.allowGrad)
                            tensors.push(this.grad);
                        if (!the_same && other?.shape) {
                            tensors.push(other);
                            if (other.allowGrad)
                                tensors.push(other.grad);
                        }
                        torus.compute(shader, [out.grad, this, ...tensors], wg.count);
                    }
                }
            }
            else{
                let cb;
                if (other === undefined) {   // Если имеется только один операнд, то всё очень просто
                    cb = new torus.CodeBuilder(
                        `// element_wise FORWARD`,
                        `const self_data = self.data;`,
                        `const out_data = out.data;`,
                        `const fn = ${$.forward};`,
                        `for (let idx = 0; idx < ${out.length}; idx++){`,
                        `    out_data[idx] = fn(self_data[idx]);`,
                        `}`
                    );
                }
                else if (!other?.shape || other.length===1 || this.length===1 || torus._shapes_are_equal(this, other)) {   // Если достаточно простой линейной выборки данных 
                    cb = new torus.CodeBuilder(
                        `// element_wise FORWARD`,
                        `const self_data = ${this.length === 1? `self.data[0]`: `self.data`};`,
                        `const other_data = ${!other?.shape? `other`: (other.length === 1? `other.data[0]`: `other.data`)};`,
                        `const out_data = out.data;`,
                        `const fn = ${$.forward};`,
                        `for (let idx = 0; idx < ${out.length}; idx++){`,
                        `    out_data[idx] = fn(self_data${this.length > 1? `[idx]`: ''}, other_data${other?.length > 1? `[idx]`: ''});`,
                        `}`
                    );
                }
                else {  // Если необходимо для каждого операнда рассчитывать собственный индекс выборки
                    cb = new torus.CodeBuilder(
                        `// element_wise FORWARD`,
                        `const self_data = self.data;`,
                        `const other_data = other.data;`,
                        `const out_data = out.data;`,
                        `const fn = ${$.forward};`,
                        `for (let idx = 0; idx < ${out.length}; idx++){`,
                        out_info.map(di => {
                            if (this_info.some(v=>v.char===di.char) && this.length!==out.length || other_info.some(v=>v.char===di.char) && other.length!==out.length)
                                return `    const idx_${di.char} = ${di.stride===1? `idx % ${di.size}`: `Math.trunc(idx / ${di.stride}) % ${di.size}`};`;
                            return '';
                        }),
                        [this_info, other_info].map((shape_info, i) => {
                            if (shape_info.length === out_info.length)
                                return  `    const idx_${i} = idx;`;
                            return `    const idx_${i} = ${shape_info.map(di => di.stride === 1? `idx_${di.char}`:`idx_${di.char} * ${di.stride}`).join(' + ')};`
                        }),
                        `    out_data[idx] = fn(self_data[idx_0], other_data[idx_1]);`,
                        `}`
                    );
                }
// >cb.code
                let fn = new Function('out', 'self', 'other', cb.code);
                out._fwd = (other)=>{
                    let src = [this];
                    if(other?.shape)
                        src.push(other);
                    fn(out, this, other);
                    return out._src(src);
                }
                if (src.some(t=>t.allowGrad)) {
                    let cb;
                    if (other === undefined) {   // Если имеется только один операнд, то всё очень просто
                        cb = new torus.CodeBuilder(
                            `// element_wise BACK`,
                            `const self_data = self.data;`,
                            `const self_grad = self.grad.data;`,
                            `const out_grad = out.grad.data;`,
                            `const fn_self = ${$.backward_0 || $.backward};`,
                            `for (let idx = 0; idx < ${out.grad.length}; idx++){`,
                            `    self_grad[idx] += out_grad[idx] * fn_self(self_data[idx]);`,
                            `}`
                        );
                    }
                    else if (!other?.shape || other.length===1 || this.length===1 || torus._shapes_are_equal(this, other)) {   // Если достаточно простой линейной выборки данных 
                        cb = new torus.CodeBuilder(
                            `// element_wise BACK`,
                            (this.length === 1)? `const x0 = self.data[0];`:
                                                 `const self_data = self.data;`,
                            (this.allowGrad)? `const self_grad = self.grad.data;`: '',
                            (!other?.shape)? `const x1 = other;`: (other.length === 1? `const x1 = other.data[0];`: `const other_data = other.data;`),
                            (other?.allowGrad)? `const other_grad = other.grad.data;`: '',
                            `const out_grad = out.grad.data;`,
                            (this.allowGrad)? `const fn_self = ${$.backward_0 || $.backward};`: '',
                            (other?.allowGrad)? `const fn_other = ${$.backward_1 || $.backward};`: '',
                            `for (let idx = 0; idx < ${out.grad.length}; idx++){`,
                            `    const grad = out_grad[idx];`,
                            (this.length > 1)? `    const x0 = self_data[idx];`: '',
                            (other?.length > 1)? `    const x1 = other_data[idx];`: '',
                            (this.allowGrad)? `    self_grad[${this.length > 1? `idx`: 0}] += grad * fn_self(x0, x1);`: '',
                            (other?.allowGrad)? `    other_grad[${other.length > 1? `idx`: 0}] += grad * fn_other(x0, x1);`: '',
                            `}`
                         );
                    }
                    else {  // Если необходимо для каждого операнда рассчитывать собственный индекс выборки
                        cb = new torus.CodeBuilder(
                            `// element_wise BACK`,
                            `const self_data = self.data;`,
                            (this.allowGrad)? `const self_grad = self.grad.data;`: '',
                            `const other_data = other.data;`,
                             (other?.allowGrad)? `const other_grad = other.grad.data;`: '',
                            `const out_grad = out.grad.data;`,
                            (this.allowGrad)? `const fn_self = ${$.backward_0 || $.backward};`: '',
                            (other?.allowGrad)? `const fn_other = ${$.backward_1 || $.backward};`: '',
                            `for (let idx = 0; idx < ${out.grad.length}; idx++){`,
                            out_info.map(di => {
                                if (this_info.some(v=>v.char===di.char) && this.length!==out.length || other_info.some(v=>v.char===di.char) && other.length!==out.length)
                                    return `    const idx_${di.char} = ${di.stride===1? `idx % ${di.size}`: `Math.trunc(idx / ${di.stride}) % ${di.size}`};`;
                                return '';
                            }),
                            [this_info, other_info].map((shape_info, i) => {
                                if (shape_info.length === out_info.length)
                                    return `    const idx_${i} = idx;`;
                                return `    const idx_${i} = ${shape_info.map(di => di.stride === 1? `idx_${di.char}`:`idx_${di.char} * ${di.stride}`).join(' + ')};`
                            }),
                            `    const grad = out_grad[idx];`,
                            `    const x0 = self_data[idx_0];`,
                            `    const x1 = other_data[idx_1];`,
                            (this.allowGrad)? `    self_grad[idx_0] += grad * fn_self(x0, x1);`: '',
                            (other?.allowGrad)? `    other_grad[idx_1] += grad * fn_other(x0, x1);`: '',
                            `}`
                        );
                    }
// >cb.code
                    src.forEach(t => {if (t.allowGrad) t.grad});    // Эта строка нужна только для отладки, точнее для детального сравнения режимов CPU и GPU.
                                                                    // Создание градиентных тензоров переносится на прямой проход как в GPU.
                    let back_fn = new Function('out', 'self', 'other', cb.code);
                    out._back = ()=>{
                        return back_fn(out, this, out.src[1] || other);
                    }
                }
            }
        }
        return out._fwd(other);
    }

In [ ]:
// 1 -- mul

torus.USE_GPU = true
torus.USE_SCALAR_CACHE = false

tensor.prototype.mul = tensor.prototype.multiply = function (other){
    const funcs = {
        forward:    '(x0, x1) => x0 * x1',
        forwardGPU: 'return x0 * x1;',
        backward_0: '(x0, x1) => x1',
        backward_1: '(x0, x1) => x0',
        backwardGPU_0: 'return x1;',
        backwardGPU_1: 'return x0;',
    };
    const out =  torus._element_wise.call(this, funcs, other);
    return out._label(`mul: (${this.shape}) * ${other?.shape? `(${other.shape})`: other}`);
}

>t1 = tensor.from([[[1,2,3],[4,5,6]], [[7,8,9],[10,11,12]]])._label('t1').p
t11 = tensor.from([[[1,2,3],[4,5,6]], [[7,8,9],[10,11,12]]])._label('t11').p
t2 = tensor.from([1,10,100])._label('t2').p
t3 = tensor.from([-1])._label('t3').p
t4 = tensor.from(-2)._label('t4').p
out = t1.mul(2)
await out.readFromGPU()
>out
await out.back(1);
> "================="
if (t1.gpuDataBuffer) { await t1.readFromGPU() }
if (t1.grad.gpuDataBuffer) { await t1.grad.readFromGPU() }
if (t2.gpuDataBuffer) { await t2.readFromGPU() }
if (t2.grad.gpuDataBuffer) { await t2.grad.readFromGPU() }
if (t3.gpuDataBuffer) { await t3.readFromGPU() }
if (t3.grad.gpuDataBuffer) { await t3.grad.readFromGPU() }
if (t4.gpuDataBuffer) { await t4.readFromGPU() }
if (t4.grad.gpuDataBuffer) { await t4.grad.readFromGPU() }
>t1
>t1.grad
>t2
>t2.grad
>t3
>t3.grad
>t4
>t4.grad